# 🦀 Day 6: Atomic Types

---

## What Are Atomics?

**Atomic types** are lock-free, thread-safe primitives. They allow safe concurrent read/write of simple values (integers, booleans) without a `Mutex`.

Use them when you need lightweight shared state — counters, flags, simple coordination.

In [ ]:
use std::sync::atomic::{AtomicBool, AtomicUsize, AtomicI64, Ordering};
use std::thread;

// AtomicBool — thread-safe boolean
let flag = AtomicBool::new(false);
flag.store(true, Ordering::SeqCst);
println!("Flag: {}", flag.load(Ordering::SeqCst));

// AtomicUsize — thread-safe usize (great for counters)
let counter = AtomicUsize::new(0);
counter.fetch_add(1, Ordering::SeqCst);
println!("Counter: {}", counter.load(Ordering::SeqCst));

// AtomicI64 — signed 64-bit integer
let value = AtomicI64::new(42);
value.fetch_sub(10, Ordering::SeqCst);
println!("Value: {}", value.load(Ordering::SeqCst));

## Memory Ordering

Atomic operations take an `Ordering` parameter. It controls visibility and ordering of memory operations:

| Ordering | Use Case |
|----------|----------|
| `Relaxed` | No ordering guarantees; counters, statistics |
| `Acquire` | Load: synchronize-after (read lock) |
| `Release` | Store: synchronize-before (write lock) |
| `SeqCst` | Sequential consistency; default when unsure |

**Rule of thumb:** Use `SeqCst` unless you know you need something else.

In [ ]:
use std::sync::atomic::{AtomicUsize, Ordering};

let counter = AtomicUsize::new(0);

// load(Ordering) — read the value
let v = counter.load(Ordering::SeqCst);
println!("Load: {}", v);

// store(value, Ordering) — write the value
counter.store(100, Ordering::SeqCst);

// fetch_add(1, Ordering) — add and return previous value
let prev = counter.fetch_add(5, Ordering::SeqCst);
println!("After fetch_add(5): prev={}, now={}", prev, counter.load(Ordering::SeqCst));

// fetch_sub(1, Ordering) — subtract and return previous value
let prev = counter.fetch_sub(3, Ordering::SeqCst);
println!("After fetch_sub(3): prev={}, now={}", prev, counter.load(Ordering::SeqCst));

## compare_exchange

`compare_exchange(current, new, success_ord, failure_ord)` — atomically: if value equals `current`, store `new` and return `Ok(prev)`. Else return `Err(actual)`.

Essential for lock-free algorithms (e.g., CAS loops).

In [ ]:
use std::sync::atomic::{AtomicUsize, Ordering};

let x = AtomicUsize::new(10);

// Try to change 10 -> 20
let result = x.compare_exchange(10, 20, Ordering::SeqCst, Ordering::SeqCst);
println!("compare_exchange(10, 20): {:?}", result);
println!("x is now: {}", x.load(Ordering::SeqCst));

// Try again — will fail because x is 20 now
let result = x.compare_exchange(10, 30, Ordering::SeqCst, Ordering::SeqCst);
println!("compare_exchange(10, 30): {:?}", result);
println!("x is still: {}", x.load(Ordering::SeqCst));

## Lock-Free Counter Example

Multiple threads incrementing a shared counter — no Mutex needed!

In [ ]:
use std::sync::atomic::{AtomicUsize, Ordering};
use std::sync::Arc;
use std::thread;

let counter = Arc::new(AtomicUsize::new(0));
let mut handles = vec![];

for _ in 0..10 {
    let c = Arc::clone(&counter);
    handles.push(thread::spawn(move || {
        for _ in 0..1000 {
            c.fetch_add(1, Ordering::SeqCst);
        }
    }));
}

for h in handles { h.join().unwrap(); }
println!("Final count: {}", counter.load(Ordering::SeqCst)); // 10000

## Spin Lock Implementation

A simple spin lock using `AtomicBool` and `compare_exchange`.

In [ ]:
use std::sync::atomic::{AtomicBool, Ordering};
use std::sync::Arc;
use std::thread;

struct SpinLock {
    locked: AtomicBool,
}

impl SpinLock {
    fn new() -> Self { SpinLock { locked: AtomicBool::new(false) } }
    fn lock(&self) {
        while self.locked.compare_exchange(false, true, Ordering::Acquire, Ordering::Relaxed).is_err() {
            // spin
        }
    }
    fn unlock(&self) {
        self.locked.store(false, Ordering::Release);
    }
}

let lock = Arc::new(SpinLock::new());
let mut handles = vec![];

for i in 0..3 {
    let l = Arc::clone(&lock);
    handles.push(thread::spawn(move || {
        l.lock();
        println!("  Thread {} acquired lock", i);
        l.unlock();
    }));
}

for h in handles { h.join().unwrap(); }
println!("Spin lock demo done!");

## When to Use Atomics vs Mutex

| Use Atomics | Use Mutex |
|-------------|----------|
| Simple counters, flags | Complex data structures |
| Lock-free algorithms | Need to hold state across multiple ops |
| Performance-critical hot paths | Arbitrary critical sections |
| Single-word operations | Multiple related fields |

Atomics: one value, one operation. Mutex: protect arbitrary code blocks.

## 🏋️ Exercises

In [ ]:
// Exercise: Implement a lock-free stack counter.
// Use AtomicUsize to track a "stack depth" that multiple threads can push/pop.
// push = fetch_add(1), pop = fetch_sub(1).
// Spawn 5 threads; each does 100 push and 100 pop. Final depth should be 0.

use std::sync::atomic::{AtomicUsize, Ordering};
use std::sync::Arc;
use std::thread;

// YOUR CODE HERE


## 🎯 Key Takeaways

1. **Atomics** are lock-free, thread-safe primitives for simple values
2. `AtomicBool`, `AtomicUsize`, `AtomicI64` — use the right type for your data
3. **Ordering**: `Relaxed`, `Acquire`, `Release`, `SeqCst` — use `SeqCst` when unsure
4. `load()`, `store()`, `fetch_add()`, `fetch_sub()`, `compare_exchange()` — core operations
5. Lock-free counters are trivial with `fetch_add`/`fetch_sub`
6. Spin locks use `compare_exchange` with Acquire/Release
7. Use atomics for single-word ops; use Mutex for complex shared state

---

**Tomorrow:** Mini-project — Multi-threaded web scraper! 🕷️